In [2]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['Agrotech'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [4]:
# --- PASO 0: LIMPIEZA ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("Limpieza lista")

# --- VARIABLES ---
NOMBRE_GRUPO = "G9-AgroTech"
INTEGRANTE = "Maximiliano Berrios"
ETIQUETA = "Diésel ENAP"

datos_finales = []
driver = None

# --- MAPA MESES ---
meses_map = {
    "01": "enero", "02": "febrero", "03": "marzo",
    "04": "abril", "05": "mayo", "06": "junio",
    "07": "julio", "08": "agosto", "09": "septiembre",
    "10": "octubre", "11": "noviembre", "12": "diciembre"
}

# --- CONFIGURACIÓN CHROME ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--headless=new")  # 👈 PEGA ESTA LÍNEA AQUÍ
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

try:
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado")

    # --- PASO 1: IR A ENAP ---
    driver.get("https://www.enap.cl/tabla-de-precios-de-paridad-historico")

    print("Cargando página...")
    time.sleep(8)

    input("Presiona ENTER para continuar...")

    # --- PASO 2: OBTENER TABLA ---
    filas = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
    print(f"Filas encontradas: {len(filas)}")

    precio_anterior = None

    for fila in filas:
        try:
            columnas = fila.find_elements(By.TAG_NAME, "td")

            if len(columnas) >= 5:
                fecha_raw = columnas[0].text.strip()
                diesel_raw = columnas[4].text.strip()

                if diesel_raw != "":
                    # --- LIMPIAR PRECIO ---
                    valor = diesel_raw.replace(".", "").replace(",", "").strip()
                    precio = float(valor) if valor.isdigit() else 0.0

                    # --- PROCESAR FECHA ---
                    partes = fecha_raw.split("-")
                    if len(partes) == 3:
                        dia, mes_num, año = partes
                        mes = meses_map.get(mes_num, mes_num)
                        año = int(año)
                    else:
                        año = 2025
                        mes = "desconocido"

                    # --- CALCULAR VARIACIÓN ---
                    variacion = 0
                    if precio_anterior is not None and precio_anterior != 0:
                        variacion = round((precio - precio_anterior) / precio_anterior, 2)

                    precio_anterior = precio

                    # --- FORMATO FINAL ---
                    datos_finales.append({
                        "grupo": NOMBRE_GRUPO,
                        "integrante": INTEGRANTE,
                        "etiqueta": ETIQUETA,
                        "año": año,
                        "mes": mes,
                        "precio": precio,
                        "variacion_decimal": variacion,
                        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
                    })

        except:
            continue

    print(f"Registros procesados: {len(datos_finales)}")

except Exception as e:
    print(f"Error: {e}")

finally:
    if driver:
        driver.quit()

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["Energia"]
    coleccion = db["Diesel_ENAP"]

    if datos_finales:
        coleccion.insert_many(datos_finales)
        print("Datos guardados en MongoDB")
    else:
        print("No se encontraron datos")

except Exception as e:
    print(f"Error MongoDB: {e}")

Limpieza lista
Navegador iniciado
Cargando página...


Presiona ENTER para continuar... 


Filas encontradas: 379
Registros procesados: 377
Datos guardados en MongoDB
